# Stage 2 Notebook 40 - Exp2KK Anchor head + ASL + AMP

**The smoking gun from NB39.** Exp2II (NB39) achieved the project's all-time best geometry: matched_iou=0.472 and oracle_f1=0.379 at epoch 20 -- THAT'S CLRKDNet territory on CULane. But pred_lanes=1536 (=192 anchors x 8 batch) -- the model predicts EVERY anchor as positive. cls is broken: with 187/192 unmatched anchors per image, plain focal alpha=0.25 is too soft on negatives. They all settle at sigmoid ~ 0.4.

Exp2KK targets THIS exact failure with a single-knob change: `cls_loss_type: focal -> asl` (Asymmetric Focal Loss with `gamma_neg=4`). ASL amplifies the gradient on confident-but-wrong negatives (high-pred unmatched anchors) by `(p_neg - clip)^gamma_neg`, forcing them down faster than focal does.

Plus two infrastructure additions (no LR/batch change so stage1 stability is unaffected):
- `train.amp.kind: bfloat16` -- mixed precision on RTX 6000, ~2x speedup with zero hyperparameter risk. Addresses GPU underutilization (3.9/95.6 GB observed).
- `eval.grad_cos_probe_interval: 50` -- joint-conflict diagnostic. Computes grad_cos (lane vs det gradient cosine on shared backbone) every 50 steps; epoch_summary now reports `train/grad_cosine_epoch_mean` so we can directly see when tasks fight.

Reference: Ben-Baruch et al. 'Asymmetric Loss For Multi-Label Classification' (CVPR 2021).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. With AMP enabled, expect ~2x speedup vs Exp2II. Estimated wall-clock: ~30 minutes for 20 epochs at 3000 samples.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp35_rmt_gca_anchor_asl_amp_joint_smoke.log
OK exp35_rmt_gca_anchor_asl_amp_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.5129 det_loss=3.0647 grad_cos=0.0264 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.500045657157898, 'gate/lane_mean': 0.5006721615791321, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp35_rmt_gca_anchor_asl_amp_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp35_rmt_gca_anchor_asl_amp_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp35_rmt_gca_anchor_asl_amp_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp35_rmt_gca_anchor_asl_amp_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve

0

## What to watch in Exp2KK training

Reference Exp2II (NB39, anchor head, focal cls): matched_iou=0.472, oracle_f1=0.379, decoded_f1=0.011, **pred_lanes=1536** (broken cls).

Pass criteria at epoch 20:
- **`pred_lanes < 200`** -- the smoking gun. ASL forced unmatched anchors below the 0.3 threshold. (NB39 had 1536 = ALL anchors above threshold.)
- **`val/lane/decoded_f1 >= 0.10`** -- 10x NB39's 0.011, because cls now ranks anchors meaningfully.
- **`val/matched_line_iou >= 0.40`** -- preserves NB39's geometric champion (don't trade it for cls).
- **`val/lane/decoded_oracle_f1 >= 0.30`** -- oracle ceiling stays high; if it drops, ASL hurt geometry too.
- `[amp] kind=bfloat16 enabled=True` log line at start.
- `train/grad_cosine_epoch_mean` is logged each epoch -- new conflict diagnostic.

Failure signals:
- pred_lanes still > 1000: ASL gamma_neg=4 still too soft. Try gamma_neg=6.
- decoded_f1 < 0.05: ASL helped reduce false positives but cls still can't separate true positives. Add OHEM (`cls_ohem_topk_per_pos: 3`).
- matched_iou drops below 0.30: ASL is interfering with geometry training. Reduce w_cls to 2.0.
- AMP causes NaN: rare on bfloat16 but possible. Set `train.amp.kind: none` to disable.